# 02: Pseudo-absence generation

A presence-only dataset cannot be fed directly into a standard classifier. Classifiers need both classes. Pseudo-absences are background points sampled from the study area and treated as the negative class.

The choice of where to sample those background points is not neutral. It determines what contrast the model is learning. Sample randomly across the whole region and the model learns what separates occupied cells from the average Aberdeenshire environment. Sample only near existing records and the model learns finer environmental discrimination but may miss the edges of the species range. This notebook generates both and maps the difference.

In [ ]:
import sqlite3
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point, MultiPoint
from shapely.ops import unary_union


In [ ]:
with sqlite3.connect('data/gbif_records.db') as conn:
    df = pd.read_sql('SELECT * FROM red_squirrel', conn)

print(f'{len(df):,} presence records loaded')
df.head()


## 1. Spatial thinning

The raw records are spatially autocorrelated: high-density clusters near Aberdeen city will dominate model fitting if left as-is. Thinning to one record per grid cell reduces that influence without discarding the geographic extent of the data.

Grid resolution is set to 0.01 degrees, roughly 700 metres at this latitude.

In [ ]:
RES = 0.01

df['grid_lon'] = (df['decimalLongitude'] // RES) * RES
df['grid_lat'] = (df['decimalLatitude']  // RES) * RES

thinned = (
    df.groupby(['grid_lon', 'grid_lat'], as_index=False)
    .first()
    .drop(columns=['grid_lon', 'grid_lat'])
)

print(f'Before thinning: {len(df):,}')
print(f'After thinning:  {len(thinned):,}')


## 2. Study area

Two study area definitions are compared here:

- **Bounding box**: the rectangle enclosing all records. Simple, but includes coastal water and areas far outside the species range.
- **Convex hull**: the smallest convex polygon containing all records. A rougher approximation of the accessible area, but less likely to include obviously unsuitable land.

The choice of study area directly shapes what the model treats as background.

In [ ]:
presence_geom = MultiPoint(
    list(zip(thinned['decimalLongitude'], thinned['decimalLatitude']))
)

bbox   = presence_geom.envelope
chull  = presence_geom.convex_hull

print('Bounding box area (sq degrees): ', round(bbox.area,  4))
print('Convex hull area  (sq degrees): ', round(chull.area, 4))


## 3. Generate pseudo-absences

Background points are sampled by rejection: random candidates are drawn from the bounding box and kept only if they fall inside the target polygon. The number of background points is set to match the thinned presence count (1:1 ratio). A fixed random seed makes the sampling reproducible.

In [ ]:
def sample_background(polygon, n, seed=42):
    rng  = np.random.default_rng(seed)
    minx, miny, maxx, maxy = polygon.bounds
    pts  = []
    while len(pts) < n:
        lons = rng.uniform(minx, maxx, n * 3)
        lats = rng.uniform(miny, maxy, n * 3)
        for lon, lat in zip(lons, lats):
            if polygon.contains(Point(lon, lat)):
                pts.append((lon, lat))
            if len(pts) == n:
                break
    return pd.DataFrame(pts, columns=['decimalLongitude', 'decimalLatitude'])

N = len(thinned)

bg_bbox  = sample_background(bbox,  N)
bg_chull = sample_background(chull, N)

print(f'Background points (bbox):  {len(bg_bbox):,}')
print(f'Background points (chull): {len(bg_chull):,}')


## 4. Compare the two strategies

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 7), facecolor='#1e1f21')
titles    = ['Thinned presences', 'Background: bounding box', 'Background: convex hull']
datasets  = [thinned, bg_bbox, bg_chull]
colours   = ['#00a68a', '#e07b39', '#7b9ee0']

for ax, data, title, col in zip(axes, datasets, titles, colours):
    ax.set_facecolor('#2a2b2d')
    ax.tick_params(colors='#aaa')
    for spine in ax.spines.values():
        spine.set_edgecolor('#555')
    ax.set_xlim(-3.8, -1.6)
    ax.set_ylim(56.75, 57.85)
    ax.set_xlabel('Longitude', color='#aaa')
    ax.set_ylabel('Latitude',  color='#aaa')
    ax.scatter(
        data['decimalLongitude'], data['decimalLatitude'],
        s=2, alpha=0.4, color=col, linewidths=0,
    )
    ax.set_title(title, color='white', pad=10)

fig.suptitle(
    'Pseudo-absence strategies compared',
    color='white', y=1.01, fontsize=13,
)
plt.tight_layout()
plt.savefig('figures/02_pseudoabsence_comparison.png', dpi=150,
            bbox_inches='tight', facecolor='#1e1f21')
plt.show()


## 5. Save to SQLite

All three datasets are saved for use in the modelling notebooks. A `presence` column marks 1 for presence records and 0 for background.

In [ ]:
thinned['presence']  = 1
bg_bbox['presence']  = 0
bg_chull['presence'] = 0

with sqlite3.connect('data/gbif_records.db') as conn:
    thinned.to_sql('presences_thinned',   conn, if_exists='replace', index=False)
    bg_bbox.to_sql('background_bbox',     conn, if_exists='replace', index=False)
    bg_chull.to_sql('background_chull',   conn, if_exists='replace', index=False)

print('Saved: presences_thinned, background_bbox, background_chull')


## What this means for modelling

The bounding box background includes sea, high moorland, and areas with no realistic chance of supporting red squirrels. A model trained against that background learns broad habitat discrimination. The convex hull background stays closer to where the species was actually recorded, so the model is pushed to learn finer environmental distinctions.

Neither is correct in an absolute sense. The bounding box version tends to produce wider, smoother predicted distributions. The convex hull version tends to produce tighter predictions that can underestimate range edges. Notebook 03 fits a model under both and maps the difference.